# 📊 Customer Churn Analysis & Prediction
## Telco Customer Churn — End-to-End Data Science Project

**Goal:** Understand why customers leave and build a model to predict churn.

**Stack:** Python · SQL · scikit-learn · matplotlib · seaborn

---


## Phase 1: Data Cleaning

In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Load dataset
df = pd.read_csv('Telco-Customer-Churn.csv')

print("=== Dataset Overview ===")
print(f"Shape: {df.shape}")
print(f"\nColumns: {list(df.columns)}")


In [ ]:
# Inspect first rows
df.head()


In [ ]:
# Check data types and missing values
print("=== Data Types & Missing Values ===")
info_df = pd.DataFrame({
    'dtype': df.dtypes,
    'null_count': df.isnull().sum(),
    'null_%': (df.isnull().sum() / len(df) * 100).round(2)
})
print(info_df[info_df['null_count'] > 0])
print(f"\nDuplicates: {df.duplicated().sum()}")


In [ ]:
# Fix TotalCharges — convert to numeric (blanks become NaN)
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')

# Fill missing TotalCharges with tenure * MonthlyCharges (logical fill)
mask = df['TotalCharges'].isnull()
df.loc[mask, 'TotalCharges'] = df.loc[mask, 'tenure'] * df.loc[mask, 'MonthlyCharges']

# Convert Churn to binary (1 = churned, 0 = stayed)
df['Churn_Binary'] = (df['Churn'] == 'Yes').astype(int)

# Convert SeniorCitizen to readable label
df['SeniorCitizen_Label'] = df['SeniorCitizen'].map({0: 'No', 1: 'Yes'})

# Remove duplicates
df.drop_duplicates(inplace=True)

print("✅ Cleaning complete!")
print(f"Final shape: {df.shape}")
print(f"Missing values remaining: {df.isnull().sum().sum()}")
print(f"Churn rate: {df['Churn_Binary'].mean()*100:.1f}%")


---
## Phase 2: Exploratory Data Analysis (EDA)

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns

sns.set_style('whitegrid')
COLORS = {'Yes': '#E24B4A', 'No': '#378ADD'}
plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.size'] = 11


In [ ]:
# ── Overall Churn Distribution ──
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Count plot
churn_counts = df['Churn'].value_counts()
bars = axes[0].bar(churn_counts.index, churn_counts.values,
                   color=[COLORS[x] for x in churn_counts.index], width=0.5, edgecolor='white')
axes[0].set_title('Overall Churn Distribution', fontweight='bold')
axes[0].set_xlabel('Churn'); axes[0].set_ylabel('Number of Customers')
for bar, val in zip(bars, churn_counts.values):
    axes[0].text(bar.get_x()+bar.get_width()/2, bar.get_height()+50, f'{val:,}', ha='center', fontweight='bold')

# Pie chart
axes[1].pie(churn_counts.values, labels=['Retained','Churned'],
            colors=['#378ADD','#E24B4A'], autopct='%1.1f%%',
            startangle=90, wedgeprops={'edgecolor':'white','linewidth':2})
axes[1].set_title('Churn Rate Breakdown', fontweight='bold')

plt.suptitle('Customer Retention Overview', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('churn_overview.png', bbox_inches='tight')
plt.show()
print(f"Churned: {churn_counts['Yes']:,} | Retained: {churn_counts['No']:,}")


In [ ]:
# ── Key Categorical Features vs Churn ──
cat_features = ['Contract', 'InternetService', 'PaymentMethod', 'SeniorCitizen_Label']
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for i, col in enumerate(cat_features):
    churn_pct = df.groupby(col)['Churn_Binary'].mean().sort_values(ascending=False) * 100
    bars = axes[i].bar(churn_pct.index, churn_pct.values,
                       color='#E24B4A', alpha=0.85, edgecolor='white')
    axes[i].set_title(f'Churn Rate by {col}', fontweight='bold')
    axes[i].set_ylabel('Churn Rate (%)')
    axes[i].set_ylim(0, churn_pct.max() * 1.25)
    axes[i].tick_params(axis='x', rotation=20)
    for bar, val in zip(bars, churn_pct.values):
        axes[i].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.5,
                     f'{val:.1f}%', ha='center', fontsize=9, fontweight='bold')

plt.suptitle('Churn Rate by Key Categories', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('churn_by_category.png', bbox_inches='tight')
plt.show()


In [ ]:
# ── Monthly Charges & Tenure vs Churn ──
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Monthly Charges distribution
for label, color in COLORS.items():
    subset = df[df['Churn'] == label]['MonthlyCharges']
    axes[0].hist(subset, bins=30, alpha=0.6, color=color, label=f'Churn={label}', edgecolor='white')
axes[0].set_title('Monthly Charges Distribution by Churn', fontweight='bold')
axes[0].set_xlabel('Monthly Charges (₹)'); axes[0].set_ylabel('Count')
axes[0].legend()

# Tenure by churn
for label, color in COLORS.items():
    subset = df[df['Churn'] == label]['tenure']
    axes[1].hist(subset, bins=30, alpha=0.6, color=color, label=f'Churn={label}', edgecolor='white')
axes[1].set_title('Tenure Distribution by Churn', fontweight='bold')
axes[1].set_xlabel('Tenure (months)'); axes[1].set_ylabel('Count')
axes[1].legend()

plt.tight_layout()
plt.savefig('charges_tenure.png', bbox_inches='tight')
plt.show()

# Business insight
avg_charge_churned = df[df['Churn']=='Yes']['MonthlyCharges'].mean()
avg_charge_stayed = df[df['Churn']=='No']['MonthlyCharges'].mean()
print(f"💡 Avg monthly charge — Churned: ₹{avg_charge_churned:.0f}  |  Retained: ₹{avg_charge_stayed:.0f}")


In [ ]:
# ── Correlation Heatmap ──
# Encode categoricals for correlation
df_enc = df.copy()
for col in df_enc.select_dtypes('object').columns:
    if col not in ['customerID']:
        df_enc[col] = pd.Categorical(df_enc[col]).codes

num_cols = ['tenure','MonthlyCharges','TotalCharges','SeniorCitizen','Churn_Binary']
corr = df_enc[num_cols].corr()

plt.figure(figsize=(7, 5))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdBu_r', center=0,
            linewidths=0.5, cbar_kws={'shrink':0.8})
plt.title('Correlation Matrix', fontweight='bold')
plt.tight_layout()
plt.savefig('correlation.png', bbox_inches='tight')
plt.show()


---
## Phase 3: SQL Analysis

In [ ]:
import sqlite3

# Load data into SQLite for SQL analysis
conn = sqlite3.connect(':memory:')
df.to_sql('customers', conn, index=False, if_exists='replace')
print("✅ Data loaded into SQLite in-memory database")

def run_query(sql, title=""):
    """Helper to run and display SQL queries"""
    result = pd.read_sql_query(sql, conn)
    if title:
        print(f"\n{'='*50}")
        print(f"  {title}")
        print('='*50)
    return result


In [ ]:
# Query 1: Overall churn rate
q1 = run_query("""
    SELECT 
        COUNT(*) AS total_customers,
        SUM(Churn_Binary) AS churned,
        ROUND(AVG(Churn_Binary)*100, 2) AS churn_rate_pct,
        SUM(CASE WHEN Churn='No' THEN 1 ELSE 0 END) AS retained
    FROM customers
""", "Q1: Overall Churn Rate")
print(q1.to_string(index=False))


In [ ]:
# Query 2: Churn rate by Contract type
q2 = run_query("""
    SELECT 
        Contract,
        COUNT(*) AS total,
        SUM(Churn_Binary) AS churned,
        ROUND(AVG(Churn_Binary)*100, 2) AS churn_rate_pct
    FROM customers
    GROUP BY Contract
    ORDER BY churn_rate_pct DESC
""", "Q2: Churn Rate by Contract Type")
print(q2.to_string(index=False))


In [ ]:
# Query 3: Revenue lost due to churn
q3 = run_query("""
    SELECT 
        ROUND(SUM(CASE WHEN Churn='Yes' THEN MonthlyCharges ELSE 0 END), 2) AS monthly_revenue_lost,
        ROUND(SUM(CASE WHEN Churn='Yes' THEN TotalCharges ELSE 0 END), 2) AS total_revenue_lost,
        COUNT(CASE WHEN Churn='Yes' THEN 1 END) AS customers_lost
    FROM customers
""", "Q3: Revenue Lost Due to Churn")
print(q3.to_string(index=False))


In [ ]:
# Query 4: Churn by Internet Service & Payment Method
q4 = run_query("""
    SELECT 
        InternetService,
        PaymentMethod,
        COUNT(*) AS total,
        ROUND(AVG(Churn_Binary)*100, 2) AS churn_rate_pct
    FROM customers
    GROUP BY InternetService, PaymentMethod
    ORDER BY churn_rate_pct DESC
    LIMIT 10
""", "Q4: Top Churn Combinations (Internet + Payment)")
print(q4.to_string(index=False))


In [ ]:
# Query 5: Churn by tenure group
q5 = run_query("""
    SELECT 
        CASE 
            WHEN tenure <= 12 THEN '0-12 months'
            WHEN tenure <= 24 THEN '13-24 months'
            WHEN tenure <= 48 THEN '25-48 months'
            ELSE '49+ months'
        END AS tenure_group,
        COUNT(*) AS total,
        ROUND(AVG(Churn_Binary)*100, 2) AS churn_rate_pct,
        ROUND(AVG(MonthlyCharges), 2) AS avg_monthly_charge
    FROM customers
    GROUP BY tenure_group
    ORDER BY churn_rate_pct DESC
""", "Q5: Churn by Tenure Group")
print(q5.to_string(index=False))


In [ ]:
# Query 6: High-risk customer segments
q6 = run_query("""
    SELECT customerID, tenure, MonthlyCharges, Contract, InternetService, PaymentMethod
    FROM customers
    WHERE Churn='No'
      AND Contract='Month-to-month'
      AND InternetService='Fiber optic'
      AND tenure < 12
    ORDER BY MonthlyCharges DESC
    LIMIT 10
""", "Q6: High-Risk Customers (Still Active but Likely to Churn)")
print(q6.to_string(index=False))


---
## Phase 4: Visualization Dashboard

In [ ]:
# ── Full Dashboard ──
fig = plt.figure(figsize=(16, 12))
fig.suptitle('Customer Churn Analysis Dashboard', fontsize=18, fontweight='bold', y=1.01)
gs = gridspec.GridSpec(3, 3, figure=fig, hspace=0.45, wspace=0.35)

# ── KPI Cards ──
total = len(df)
churned = df['Churn_Binary'].sum()
churn_rate = df['Churn_Binary'].mean() * 100
rev_lost = df[df['Churn']=='Yes']['MonthlyCharges'].sum()

ax_kpi = fig.add_subplot(gs[0, :])
ax_kpi.axis('off')
kpis = [
    (f"{total:,}", "Total Customers", '#378ADD'),
    (f"{churned:,}", "Churned Customers", '#E24B4A'),
    (f"{churn_rate:.1f}%", "Churn Rate", '#E24B4A'),
    (f"₹{rev_lost:,.0f}", "Monthly Revenue Lost", '#E85D24'),
    (f"₹{df['MonthlyCharges'].mean():.0f}", "Avg Monthly Charge", '#639922'),
]
for i, (val, label, color) in enumerate(kpis):
    x = 0.05 + i * 0.19
    ax_kpi.add_patch(plt.Rectangle((x, 0.05), 0.16, 0.88, transform=ax_kpi.transAxes,
                                    facecolor=color+'22', edgecolor=color, linewidth=1.5))
    ax_kpi.text(x+0.08, 0.62, val, transform=ax_kpi.transAxes,
                ha='center', fontsize=16, fontweight='bold', color=color)
    ax_kpi.text(x+0.08, 0.25, label, transform=ax_kpi.transAxes,
                ha='center', fontsize=9, color='#555')

# ── Plot 1: Churn by Contract ──
ax1 = fig.add_subplot(gs[1, 0])
cp = df.groupby('Contract')['Churn_Binary'].mean().sort_values(ascending=False)*100
bars = ax1.bar(cp.index, cp.values, color=['#E24B4A','#E8C46A','#378ADD'], edgecolor='white')
ax1.set_title('Churn by Contract', fontweight='bold')
ax1.set_ylabel('Churn Rate (%)')
ax1.tick_params(axis='x', rotation=15, labelsize=8)
for b, v in zip(bars, cp.values):
    ax1.text(b.get_x()+b.get_width()/2, b.get_height()+0.5, f'{v:.1f}%', ha='center', fontsize=8, fontweight='bold')

# ── Plot 2: Churn by Internet ──
ax2 = fig.add_subplot(gs[1, 1])
ip = df.groupby('InternetService')['Churn_Binary'].mean().sort_values(ascending=False)*100
bars2 = ax2.bar(ip.index, ip.values, color=['#E24B4A','#E8C46A','#378ADD'], edgecolor='white')
ax2.set_title('Churn by Internet Service', fontweight='bold')
ax2.set_ylabel('Churn Rate (%)')
for b, v in zip(bars2, ip.values):
    ax2.text(b.get_x()+b.get_width()/2, b.get_height()+0.5, f'{v:.1f}%', ha='center', fontsize=8, fontweight='bold')

# ── Plot 3: Tenure distribution ──
ax3 = fig.add_subplot(gs[1, 2])
for label, color in COLORS.items():
    ax3.hist(df[df['Churn']==label]['tenure'], bins=25, alpha=0.6, color=color, label=label, edgecolor='white')
ax3.set_title('Tenure by Churn', fontweight='bold')
ax3.set_xlabel('Tenure (months)'); ax3.legend(title='Churn', fontsize=8)

# ── Plot 4: Monthly Charges Box ──
ax4 = fig.add_subplot(gs[2, 0])
groups = [df[df['Churn']=='No']['MonthlyCharges'], df[df['Churn']=='Yes']['MonthlyCharges']]
bp = ax4.boxplot(groups, labels=['Retained','Churned'], patch_artist=True, notch=True)
bp['boxes'][0].set_facecolor('#378ADD44'); bp['boxes'][1].set_facecolor('#E24B4A44')
ax4.set_title('Monthly Charges Distribution', fontweight='bold')
ax4.set_ylabel('Monthly Charges (₹)')

# ── Plot 5: Payment Method ──
ax5 = fig.add_subplot(gs[2, 1])
pp = df.groupby('PaymentMethod')['Churn_Binary'].mean().sort_values()*100
short = [p.replace(' (automatic)','*').replace('Electronic ','E. ').replace('Mailed ','M. ').replace('Bank ','Bank ') for p in pp.index]
ax5.barh(short, pp.values, color='#E24B4A', alpha=0.8, edgecolor='white')
ax5.set_title('Churn by Payment Method', fontweight='bold')
ax5.set_xlabel('Churn Rate (%)')

# ── Plot 6: Churn by Tenure Group ──
ax6 = fig.add_subplot(gs[2, 2])
df['tenure_group'] = pd.cut(df['tenure'], bins=[0,12,24,48,72],
                             labels=['0-12m','13-24m','25-48m','49+m'])
tg = df.groupby('tenure_group', observed=True)['Churn_Binary'].mean()*100
ax6.plot(tg.index.astype(str), tg.values, 'o-', color='#E24B4A', linewidth=2, markersize=8)
ax6.fill_between(range(len(tg)), tg.values, alpha=0.15, color='#E24B4A')
ax6.set_title('Churn Rate by Tenure Group', fontweight='bold')
ax6.set_ylabel('Churn Rate (%)')
ax6.set_xticks(range(len(tg))); ax6.set_xticklabels(tg.index.astype(str))

plt.savefig('churn_dashboard.png', bbox_inches='tight', dpi=150)
plt.show()
print("✅ Dashboard saved as churn_dashboard.png")


---
## Phase 5: Machine Learning — Churn Prediction Model

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import (classification_report, confusion_matrix,
                             roc_auc_score, roc_curve, ConfusionMatrixDisplay)

# ── Feature Engineering ──
df_ml = df.copy()
drop_cols = ['customerID', 'Churn', 'tenure_group', 'SeniorCitizen_Label', 'TotalCharges']
df_ml.drop(columns=drop_cols, inplace=True, errors='ignore')

# Encode all categorical columns
le = LabelEncoder()
for col in df_ml.select_dtypes('object').columns:
    df_ml[col] = le.fit_transform(df_ml[col])

X = df_ml.drop('Churn_Binary', axis=1)
y = df_ml['Churn_Binary']

print("Features used:", list(X.columns))
print(f"Dataset size: {X.shape}")


In [ ]:
# ── Train/Test Split + Scale ──
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

# ── Train Logistic Regression ──
model = LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced')
model.fit(X_train_sc, y_train)

y_pred = model.predict(X_test_sc)
y_prob = model.predict_proba(X_test_sc)[:, 1]

print("✅ Model trained!")
print(f"Training samples: {len(X_train)} | Test samples: {len(X_test)}")


In [ ]:
# ── Model Evaluation ──
print("=" * 50)
print("  MODEL PERFORMANCE REPORT")
print("=" * 50)
print(classification_report(y_test, y_pred, target_names=['Retained','Churned']))
print(f"ROC-AUC Score: {roc_auc_score(y_test, y_prob):.3f}")


In [ ]:
# ── Confusion Matrix + ROC Curve ──
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(cm, display_labels=['Retained','Churned'])
disp.plot(ax=axes[0], colorbar=False, cmap='Blues')
axes[0].set_title('Confusion Matrix', fontweight='bold')

# ROC Curve
fpr, tpr, _ = roc_curve(y_test, y_prob)
auc = roc_auc_score(y_test, y_prob)
axes[1].plot(fpr, tpr, color='#E24B4A', linewidth=2, label=f'AUC = {auc:.3f}')
axes[1].plot([0,1],[0,1],'--', color='gray', linewidth=1)
axes[1].fill_between(fpr, tpr, alpha=0.1, color='#E24B4A')
axes[1].set_title('ROC Curve', fontweight='bold')
axes[1].set_xlabel('False Positive Rate'); axes[1].set_ylabel('True Positive Rate')
axes[1].legend(fontsize=11)

plt.tight_layout()
plt.savefig('model_evaluation.png', bbox_inches='tight')
plt.show()


In [ ]:
# ── Feature Importance ──
feature_imp = pd.DataFrame({
    'Feature': X.columns,
    'Coefficient': model.coef_[0],
    'Abs_Coeff': abs(model.coef_[0])
}).sort_values('Abs_Coeff', ascending=False).head(10)

plt.figure(figsize=(9, 5))
colors = ['#E24B4A' if c > 0 else '#378ADD' for c in feature_imp['Coefficient']]
bars = plt.barh(feature_imp['Feature'], feature_imp['Coefficient'], color=colors, edgecolor='white')
plt.axvline(0, color='black', linewidth=0.8, linestyle='--')
plt.title('Top 10 Feature Importances (Logistic Regression)', fontweight='bold')
plt.xlabel('Coefficient Value (Red = increases churn, Blue = decreases churn)')
plt.tight_layout()
plt.savefig('feature_importance.png', bbox_inches='tight')
plt.show()


In [ ]:
# ── Predict for a New Customer ──
def predict_churn(customer_dict):
    """Predict churn probability for a new customer"""
    sample = pd.DataFrame([customer_dict])
    # Encode
    for col in sample.select_dtypes('object').columns:
        sample[col] = le.transform(sample[col])
    sample = sample.reindex(columns=X.columns, fill_value=0)
    sample_sc = scaler.transform(sample)
    prob = model.predict_proba(sample_sc)[0][1]
    pred = 'LIKELY TO CHURN ⚠️' if prob > 0.5 else 'LIKELY TO STAY ✅'
    print(f"Churn Probability: {prob*100:.1f}%  →  {pred}")
    return prob

# Example: High-risk customer
print("--- High-Risk Customer ---")
predict_churn({
    'gender': 0, 'SeniorCitizen': 1, 'Partner': 0, 'Dependents': 0,
    'tenure': 3, 'PhoneService': 1, 'MultipleLines': 1, 'InternetService': 2,
    'OnlineSecurity': 0, 'OnlineBackup': 0, 'DeviceProtection': 0, 'TechSupport': 0,
    'StreamingTV': 1, 'StreamingMovies': 1, 'Contract': 0, 'PaperlessBilling': 1,
    'PaymentMethod': 1, 'MonthlyCharges': 95.0, 'Churn_Binary': 0
})

# Example: Low-risk customer
print("\n--- Low-Risk Customer ---")
predict_churn({
    'gender': 1, 'SeniorCitizen': 0, 'Partner': 1, 'Dependents': 1,
    'tenure': 55, 'PhoneService': 1, 'MultipleLines': 0, 'InternetService': 1,
    'OnlineSecurity': 1, 'OnlineBackup': 1, 'DeviceProtection': 1, 'TechSupport': 1,
    'StreamingTV': 0, 'StreamingMovies': 0, 'Contract': 2, 'PaperlessBilling': 0,
    'PaymentMethod': 2, 'MonthlyCharges': 55.0, 'Churn_Binary': 0
})


---
## 📌 Business Insights & Recommendations

### Key Findings:
1. **Month-to-month contracts** have the highest churn rate — customers without long-term commitment leave easily
2. **Fiber optic internet users** churn more — possibly due to high pricing or unmet expectations
3. **Electronic check payment** correlates with higher churn — may indicate less financially stable customers
4. **New customers (0–12 months)** are the most at-risk — the first year is critical for retention
5. **Customers with TechSupport** churn significantly less — support drives loyalty

### Recommendations:
| Segment | Action | Expected Impact |
|---|---|---|
| Month-to-month, Fiber users | Offer 3-month loyalty discount | Reduce churn by ~15% |
| First 6 months tenure | Personalized onboarding program | Reduce early churn |
| Electronic check users | Incentivize auto-pay switch | Reduce churn + improve cash flow |
| High monthly charge (>₹80) | Offer bundle deals | Retain high-value customers |
| Senior citizens | Dedicated support line | Improve satisfaction & retention |
